# Week 9 · Resource levelling: the parallel heuristic and the exact optimum

IMSE 456/656 · Program and Project Management · Larson & Gray, Chapter 8

This notebook takes a project network with one renewable resource, computes the early-start load, levels the schedule two ways and reports the difference:

1. **Parallel method**, the chapter's heuristic: period by period, priority to minimum slack, then shortest duration, then lowest ID.
2. **Exact optimum**: every precedence-feasible activity order is tried through a serial schedule generator; for larger networks a constraint solver (OR-Tools CP-SAT) does the same search exactly.

**How to use it.** Run the cells top to bottom (Runtime → Run all). Edit only the cells marked **EDIT**: the activity table, the resource limit and the penalty terms. Everything else recomputes. The default data is the conference from the Week 9 notes; replace it with your own project to answer the studio's questions.

| Column | Meaning |
|:--|:--|
| `id` | A short label; predecessors refer to it |
| `name` | What the activity is |
| `dur` | Duration in workdays, a positive integer |
| `pred` | List of immediate predecessor ids, finish-to-start, zero lag |
| `need` | Units of the resource the activity uses on every day it runs |
| `cost` | Direct cost, spread evenly over the activity's days |

Assumptions, as in the chapter: one resource type, a constant limit, no splitting once an activity starts, every activity runs at its full demand for its whole duration.

In [ ]:
# ---------------------------------------------------------------- EDIT · the project
ACTIVITIES = [
    # id, name,                        dur, predecessors, need, cost
    ("A", "Approve event brief",         2, [],           1,  2000),
    ("B", "Contract venue",              5, ["A"],        2, 30000),
    ("C", "Draft program",               6, ["A"],        1,  6000),
    ("D", "Configure registration",      4, ["B"],        1,  8000),
    ("E", "Confirm speakers",            8, ["C"],        2, 12000),
    ("F", "Recruit exhibitors",          5, ["C"],        2,  2500),
    ("G", "Publish program",             3, ["D", "E"],   1,  3000),
    ("H", "Prepare on-site materials",   4, ["F", "G"],   2,  4000),
    ("I", "Complete rehearsal",          2, ["H"],        2,  2500),
]
RESOURCE_NAME = "coordinators"
LIMIT = 2                 # units of the resource available on every day

# Reporting
SPONSOR_DAY = 32          # the date the sponsor wants, as a workday number (0 = project start)
PENALTY_PER_DAY = 1500    # dollars for every workday the finish runs past SPONSOR_DAY
WEEK_LENGTH = 5           # workdays per reporting week for the time-phased budget

# Scenario 3 · a temporary extra unit of the resource (None = no loan)
LOAN = None               # e.g. (19, 23, 1): one extra unit on workdays 19 to 23 inclusive (0 = project start)
LOAN_COST_PER_DAY = 600   # dollars per extra unit per day of the loan

## 1 · Check the data and run CPM

The network must be acyclic and every predecessor must exist. CPM gives each activity its early and late dates with the resource ignored; the late finish is what the parallel method's priority rule needs.

In [ ]:
import itertools, math
from collections import OrderedDict

ACT = OrderedDict((a[0], {"name": a[1], "dur": int(a[2]), "pred": list(a[3]), "need": int(a[4]), "cost": float(a[5])}) for a in ACTIVITIES)
IDS = list(ACT)

def cap(t):
    # units available on workday t: LIMIT, plus the loan inside its window
    if LOAN and LOAN[0] <= t <= LOAN[1]: return LIMIT + LOAN[2]
    return LIMIT
MAXCAP = LIMIT + (LOAN[2] if LOAN else 0)
LIMIT_TEXT = f"{LIMIT}" + (f" (+{LOAN[2]} on days {LOAN[0]}–{LOAN[1]})" if LOAN else "")
LOAN_COST = (LOAN[1] - LOAN[0] + 1) * LOAN[2] * LOAN_COST_PER_DAY if LOAN else 0

# validation
for a, x in ACT.items():
    assert x["dur"] > 0, f"{a}: duration must be positive"
    assert x["need"] > 0, f"{a}: need must be positive"
    for p in x["pred"]:
        assert p in ACT, f"{a}: predecessor {p} is not an activity"
    assert x["need"] <= MAXCAP, f"{a} needs {x['need']} {RESOURCE_NAME} but at most {MAXCAP} exist: it can never run"

# topological order (also detects cycles)
order, seen = [], set()
def visit(a, stack=()):
    if a in stack: raise ValueError(f"cycle through {a}")
    if a in seen: return
    for p in ACT[a]["pred"]: visit(p, stack + (a,))
    seen.add(a); order.append(a)
for a in IDS: visit(a)
IDS = order
SUCC = {a: [s for s in IDS if a in ACT[s]["pred"]] for a in IDS}

def cpm():
    es, ef = {}, {}
    for a in IDS:
        es[a] = max([ef[p] for p in ACT[a]["pred"]], default=0); ef[a] = es[a] + ACT[a]["dur"]
    T = max(ef.values()); lf, ls = {}, {}
    for a in reversed(IDS):
        lf[a] = min([ls[s] for s in SUCC[a]], default=T); ls[a] = lf[a] - ACT[a]["dur"]
    return es, ef, ls, lf, T
ES, EF, LS, LF, T_CPM = cpm()

import pandas as pd
cpm_table = pd.DataFrame({"activity": [ACT[a]["name"] for a in IDS], "dur": [ACT[a]["dur"] for a in IDS],
                          "pred": [", ".join(ACT[a]["pred"]) or "—" for a in IDS], RESOURCE_NAME: [ACT[a]["need"] for a in IDS],
                          "ES": [ES[a] for a in IDS], "EF": [EF[a] for a in IDS], "LS": [LS[a] for a in IDS], "LF": [LF[a] for a in IDS],
                          "slack": [LS[a] - ES[a] for a in IDS]}, index=IDS)
print(f"Network duration with unlimited {RESOURCE_NAME}: {T_CPM} workdays")
print("Critical path:", "–".join(a for a in IDS if LS[a] == ES[a]))
cpm_table

## 2 · The resource loading chart

Every activity at its early start; the resource used each day added up. Days above the limit are the problem the rest of the notebook solves.

In [ ]:
import matplotlib.pyplot as plt

def load(start):
    H = max(start[a] + ACT[a]["dur"] for a in IDS); L = [0] * H
    for a in IDS:
        for t in range(start[a], start[a] + ACT[a]["dur"]): L[t] += ACT[a]["need"]
    return L

def finish(start): return max(start[a] + ACT[a]["dur"] for a in IDS)

def plot_load(start, title, ax=None):
    L = load(start); ax = ax or plt.gca()
    H = len(L)
    for t in range(H):
        y = 0
        for a in sorted(IDS, key=lambda a: -ACT[a]["need"]):
            if start[a] <= t < start[a] + ACT[a]["dur"]:
                h = ACT[a]["need"]; over = L[t] > cap(t)
                ax.bar(t + 0.5, h, bottom=y, width=0.95, color="#B5543A" if over else ("#0F5A4B" if h >= 2 else "#8FB8AD"), edgecolor="white", linewidth=0.6)
                ax.text(t + 0.5, y + h / 2, a, ha="center", va="center", fontsize=7, color="white", fontweight="bold"); y += h
    ax.step(range(H + 1), [cap(t) for t in range(H + 1)], where="post", color="#B5543A", ls="--", lw=1.2)
    ax.text(H + 0.3, cap(H), f"{cap(H)} available", color="#B5543A", va="center", fontsize=9)
    ax.set_xlim(0, H + 3); ax.set_ylim(0, max(max(L), MAXCAP) + 1); ax.set_xlabel("workday"); ax.set_ylabel(RESOURCE_NAME)
    ax.set_title(f"{title} · finish day {finish(start)} · peak {max(L)}", loc="left", fontsize=10, color="#0F5A4B", fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

fig, ax = plt.subplots(figsize=(11, 3.2)); plot_load(ES, "Early-start schedule"); plt.show()
L = load(ES)
print("Load by day:", L)
print("Days over the limit:", [t for t, v in enumerate(L) if v > cap(t)] or "none")
if all(v <= cap(t) for t, v in enumerate(L)): print("The early-start schedule already fits. Levelling changes nothing; try a smaller LIMIT.")

## 3 · The parallel method

The chapter's heuristic, as written in Section 8.4. At each period it lists the eligible activities (all predecessors finished, not yet started), computes each one's slack as *late finish − current period − duration*, sorts by slack, then duration, then id, and starts as many as fit under the limit. Everything else waits for the next period. The log below is the record a student keeps by hand in Studio 9.

In [ ]:
def parallel_method():
    # period by period; an eligible activity starts only if it fits under the limit on every day it will run
    H = sum(ACT[a]["dur"] for a in IDS) + (LOAN[1] + 1 if LOAN else 0) + 1
    start, L, log, last = {}, [0] * (H + 1), [], None
    for t in range(H):
        if len(start) == len(IDS): break
        running = [a for a in start if start[a] + ACT[a]["dur"] > t]
        eligible = [a for a in IDS if a not in start and all(p in start and start[p] + ACT[p]["dur"] <= t for p in ACT[a]["pred"])]
        eligible.sort(key=lambda a: (LF[a] - t - ACT[a]["dur"], ACT[a]["dur"], a))
        started, delayed = [], []
        for a in eligible:
            d, n = ACT[a]["dur"], ACT[a]["need"]
            if all(L[u] + n <= cap(u) for u in range(t, t + d)):
                start[a] = t; started.append(a)
                for u in range(t, t + d): L[u] += n
            else: delayed.append(a)
        if eligible and (started or set(eligible) != last):
            log.append({"period": t, "running": ", ".join(running) or "—", "available": cap(t),
                        "eligible (slack, need)": ", ".join(f"{a} ({LF[a] - t - ACT[a]['dur']}, {ACT[a]['need']})" for a in eligible),
                        "started": ", ".join(started) or "—", "delayed": ", ".join(delayed) or "—", "in use after": L[t]})
        last = set(eligible) - set(started)
    return start, pd.DataFrame(log)

HEUR, heur_log = parallel_method()
print(f"Parallel method finish: day {finish(HEUR)}  (network: {T_CPM})")
print("Start days:", dict(HEUR))
heur_log

Read the log for the decision that costs the most. In the conference with two coordinators it is period 2: C has zero slack and needs one person, B has five days of slack and needs two. Minimum slack starts C, and B then cannot start until C finishes, because B needs both people.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2)); plot_load(HEUR, "Parallel method"); plt.show()

## 4 · The exact optimum

A levelled schedule is defined by the order in which activities are started. For a regular objective such as the finish day, trying every **precedence-feasible order** through a serial schedule generator (start each activity at the earliest day where its predecessors are done and the resource fits) is guaranteed to include an optimal schedule (Kolisch, 1996). With nine activities there are only a few dozen such orders. The count grows factorially, so for more than about twelve activities the cell below switches to OR-Tools CP-SAT, an exact constraint solver, and installs it if needed.

In [ ]:
def serial_sgs(order):
    start, L = {}, [0] * (sum(ACT[a]["dur"] for a in IDS) + (LOAN[1] + 1 if LOAN else 0) + 2)
    for a in order:
        t = max([start[p] + ACT[p]["dur"] for p in ACT[a]["pred"]], default=0)
        while any(L[u] + ACT[a]["need"] > cap(u) for u in range(t, t + ACT[a]["dur"])): t += 1
        start[a] = t
        for u in range(t, t + ACT[a]["dur"]): L[u] += ACT[a]["need"]
    return start

def feasible_orders():
    # generate precedence-feasible permutations without building the infeasible ones
    def rec(prefix, remaining):
        if not remaining: yield tuple(prefix); return
        for a in remaining:
            if all(p in prefix for p in ACT[a]["pred"]):
                yield from rec(prefix + [a], [b for b in remaining if b != a])
    yield from rec([], list(IDS))

def optimum_by_enumeration():
    best, n = None, 0
    for order in feasible_orders():
        n += 1; s = serial_sgs(order); f = finish(s)
        if best is None or f < best[0]: best = (f, s, order)
    return best[1], best[2], n

def optimum_by_cpsat():
    try:
        from ortools.sat.python import cp_model
    except ImportError:
        import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ortools"], check=True)
        from ortools.sat.python import cp_model
    m = cp_model.CpModel(); H = sum(ACT[a]["dur"] for a in IDS) + (LOAN[1] + 1 if LOAN else 0)
    st = {a: m.NewIntVar(0, H, f"s_{a}") for a in IDS}
    iv = {a: m.NewIntervalVar(st[a], ACT[a]["dur"], st[a] + ACT[a]["dur"], f"i_{a}") for a in IDS}
    for a in IDS:
        for p in ACT[a]["pred"]: m.Add(st[a] >= st[p] + ACT[p]["dur"])
    blocks, demands = [], []
    if LOAN:  # capacity is MAXCAP; fixed blocks use the extra units outside the loan window
        for lo, hi in ((0, LOAN[0]), (LOAN[1] + 1, H + 1)):
            if hi > lo: blocks.append(m.NewIntervalVar(lo, hi - lo, hi, f"cap_{lo}")); demands.append(LOAN[2])
    m.AddCumulative([iv[a] for a in IDS] + blocks, [ACT[a]["need"] for a in IDS] + demands, MAXCAP)
    mk = m.NewIntVar(0, H, "makespan"); m.AddMaxEquality(mk, [st[a] + ACT[a]["dur"] for a in IDS]); m.Minimize(mk)
    solver = cp_model.CpSolver(); solver.parameters.max_time_in_seconds = 60
    status = solver.Solve(m); assert status in (cp_model.OPTIMAL, cp_model.FEASIBLE), "no schedule found"
    return {a: solver.Value(st[a]) for a in IDS}, "CP-SAT " + ("optimal" if status == cp_model.OPTIMAL else "feasible, time limit hit"), None

if len(IDS) <= 12:
    OPT, how, n_orders = optimum_by_enumeration()
    method = f"enumeration of {n_orders} precedence-feasible orders; best order {'–'.join(how)}"
else:
    OPT, method, _ = optimum_by_cpsat()

print(f"Optimum finish: day {finish(OPT)}  (parallel method: {finish(HEUR)}, network: {T_CPM})")
print("Method:", method)
print("Start days:", dict(OPT))
fig, ax = plt.subplots(figsize=(11, 3.2)); plot_load(OPT, "Optimum"); plt.show()

## 5 · Check both schedules

Three checks, in this order: logic, limit, no splitting. The third holds by construction here; the first two are tested.

In [ ]:
def check(start, label):
    logic = all(start[a] >= start[p] + ACT[p]["dur"] for a in IDS for p in ACT[a]["pred"])
    limit_ok = all(v <= cap(t) for t, v in enumerate(load(start)))
    print(f"{label:16} logic {'ok' if logic else 'BROKEN'} · limit {'ok' if limit_ok else 'EXCEEDED'} · finish day {finish(start)}")
    return logic and limit_ok
assert check(HEUR, "parallel method") and check(OPT, "optimum")

## 6 · Report

Side by side: the two schedules, the Gantt charts, the price of the heuristic in days and dollars, and the time-phased budget each schedule implies. The budget spreads each activity's cost evenly over its days and totals by reporting week.

In [ ]:
def gantt(start, title, ax):
    rows = sorted(IDS, key=lambda a: (start[a], a))
    for i, a in enumerate(rows):
        ax.barh(i, ACT[a]["dur"], left=start[a], color="#0F5A4B" if ACT[a]["need"] >= 2 else "#8FB8AD", height=0.6)
        ax.text(start[a] + ACT[a]["dur"] / 2, i, str(ACT[a]["need"]), ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows); ax.invert_yaxis()
    ax.axvline(T_CPM, color="#627571", ls="--", lw=1); ax.axvline(finish(start), color="#B5543A", lw=1.6)
    ax.text(finish(start) + 0.3, 0, f"day {finish(start)}", color="#B5543A", fontsize=9, fontweight="bold")
    ax.set_title(title, loc="left", fontsize=10, color="#0F5A4B", fontweight="bold"); ax.set_xlabel("workday"); ax.spines[["top", "right"]].set_visible(False)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
gantt(HEUR, f"Parallel method, {LIMIT_TEXT} {RESOURCE_NAME}: finish day {finish(HEUR)}", axes[0])
gantt(OPT, f"Optimum, {LIMIT_TEXT} {RESOURCE_NAME}: finish day {finish(OPT)}", axes[1]); plt.tight_layout(); plt.show()

gap = finish(HEUR) - finish(OPT)
pen = lambda f: max(0, f - SPONSOR_DAY) * PENALTY_PER_DAY
summary = pd.DataFrame({
    "schedule": ["Network (unlimited)", "Parallel method", "Optimum"],
    "finish day": [T_CPM, finish(HEUR), finish(OPT)],
    f"days past day {SPONSOR_DAY}": [max(0, T_CPM - SPONSOR_DAY), max(0, finish(HEUR) - SPONSOR_DAY), max(0, finish(OPT) - SPONSOR_DAY)],
    "penalty $": [pen(T_CPM), pen(finish(HEUR)), pen(finish(OPT))],
    "loan $": [0, LOAN_COST, LOAN_COST],
    "penalty + loan $": [pen(T_CPM), pen(finish(HEUR)) + LOAN_COST, pen(finish(OPT)) + LOAN_COST],
    "peak load": [max(load(ES)), max(load(HEUR)), max(load(OPT))],
}).set_index("schedule")
print(f"Limit: {LIMIT_TEXT} {RESOURCE_NAME}" + (f"; loan cost ${LOAN_COST:,}" if LOAN else ""))
print(f"Price of the heuristic: {gap} workday(s), ${pen(finish(HEUR)) - pen(finish(OPT)):,} at ${PENALTY_PER_DAY:,} a day past day {SPONSOR_DAY}")
summary

In [ ]:
def phased(start):
    H = finish(start); day = [0.0] * H
    for a in IDS:
        for t in range(start[a], start[a] + ACT[a]["dur"]): day[t] += ACT[a]["cost"] / ACT[a]["dur"]
    weeks = [sum(day[i:i + WEEK_LENGTH]) for i in range(0, H, WEEK_LENGTH)]
    return weeks

W = max(len(phased(s)) for s in (ES, HEUR, OPT))
budget = pd.DataFrame({name: (phased(s) + [0.0] * W)[:W] for name, s in (("Early start", ES), ("Parallel method", HEUR), ("Optimum", OPT))},
                      index=[f"Wk {w + 1}" for w in range(W)])
print("Weekly planned value ($)"); display(budget.round(0).astype(int))
print("Cumulative planned value ($)"); display(budget.cumsum().round(0).astype(int))
assert all(abs(budget[c].sum() - sum(x["cost"] for x in ACT.values())) < 1e-6 for c in budget.columns), "a week is missing"

fig, ax = plt.subplots(figsize=(9, 3.5))
for c, color, ls in (("Early start", "#627571", "--"), ("Parallel method", "#8FB8AD", "-"), ("Optimum", "#0F5A4B", "-")):
    ax.plot(range(1, W + 1), budget[c].cumsum(), color=color, ls=ls, marker="o", ms=3, label=c)
ax.set_xlabel("reporting week"); ax.set_ylabel("cumulative planned value, $"); ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
ax.set_title("The same budget, three baselines", loc="left", fontsize=10, color="#0F5A4B", fontweight="bold"); plt.show()

## 7 · Save the results

One table with both schedules' start and finish days, ready to paste into your studio report.

In [ ]:
result = pd.DataFrame({"activity": [ACT[a]["name"] for a in IDS], "dur": [ACT[a]["dur"] for a in IDS], RESOURCE_NAME: [ACT[a]["need"] for a in IDS],
                       "ES": [ES[a] for a in IDS], "heuristic start": [HEUR[a] for a in IDS], "heuristic finish": [HEUR[a] + ACT[a]["dur"] for a in IDS],
                       "optimum start": [OPT[a] for a in IDS], "optimum finish": [OPT[a] + ACT[a]["dur"] for a in IDS]}, index=IDS)
result.to_csv("week-09-levelling-results.csv")
print("Saved week-09-levelling-results.csv (Files panel on the left in Colab)")
result

## 8 · Try it

- Set `LIMIT = 3` and run again. Does the parallel method still lose to the optimum? (In the notes' conference it does not.)
- **A five-day loan.** Keep `LIMIT = 2` and set `LOAN = (19, 23, 1)`: one extra coordinator on workdays 19 to 23. Run all and read the summary's *penalty + loan* column. Then move the window, for example `(2, 6, 1)` or `(10, 14, 1)`, and find the window with the lowest total.
- Set `LIMIT = 2` and change B's `need` to 1. Where does the rule go wrong now, if anywhere?
- Replace the table with the chapter's seven-activity example (Figure 8.4): activities 1–7, programmers 2, 2, 2, 1, 1, 1, 1, durations 2, 6, 4, 2, 2, 4, 2, predecessors 1 → 2, 3, 4; 3 → 5, 6; 2, 5, 6 → 7; `LIMIT = 3`. The parallel method should finish in 14 periods.
- Enter your own project. If it has more than twelve activities the optimum cell installs OR-Tools and solves exactly; report whether the solver proved optimality or hit the time limit.

Reference: Kolisch, R. (1996). Serial and parallel resource-constrained project scheduling methods revisited: Theory and computation. *European Journal of Operational Research*, 90(2), 320–333.